# Вариант №19: размеченные позиции чеков

Источник: [ODS.AI, соревнование по структуризации чеков ОФД](https://ods.ai/competitions/nlp-receipts/data), файл `train_supervised_dataset.csv` является размеченным датасетом с чековыми позициями. Каждая строка — позиция чека: `id`: идентификатор, `name`: исходное наименование, `good` и `brand` — нормализованные товар и бренд. Пустая разметка товара или бренда допустима.


In [27]:
%pip -q install "psycopg[binary]" certifi pandas



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Скачать CSV

Прямая публичная ссылка указана на странице соревнования.

In [28]:
from pathlib import Path
from urllib.request import urlopen
import ssl
import certifi

url = "https://storage.yandexcloud.net/ds-ods/files/materials/8164c5a7/train_supervised_dataset.csv"
csv_file = Path("train_supervised_dataset.csv")
with urlopen(url, context=ssl.create_default_context(cafile=certifi.where())) as response:
    last_modified = response.headers.get("Last-Modified")
    csv_file.write_bytes(response.read())
print(f"Скачано: {csv_file} ({csv_file.stat().st_size:,} байт); Last-Modified: {last_modified}")

Скачано: train_supervised_dataset.csv (2,379,415 байт); Last-Modified: Thu, 01 Jun 2023 14:42:40 GMT


In [29]:
import pandas as pd

df = pd.read_csv(csv_file)
print("Размер:", df.shape)
print("Столбцы:", df.columns.tolist())
df.head()

Размер: (25000, 4)
Столбцы: ['id', 'name', 'good', 'brand']


,id,name,good,brand
0,0,Petmax Бантик леопард с красн розой 2шт,бантик,petmax
1,1,87191 Бусы для елки шарики_87191,бусы,NaN
2,2,Футболка Piazza Italia WR011446881,футболка,piazza italia
3,3,7) YI572-03X-ONE ЗАКОЛКА ДЛЯ ВОЛОС ДЛЯ ДЕВОЧКИ,заколка,NaN
4,4,Одежда (вес) 1500,одежда,NaN


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      25000 non-null  int64
 1   name    25000 non-null  str  
 2   good    24181 non-null  str  
 3   brand   16495 non-null  str  
dtypes: int64(1), str(3)
memory usage: 781.4 KB


## 2. Создать таблицу и загрузить данные

Пустые `good` и `brand` при `COPY CSV` становятся `NULL`. Повторный запуск заменяет строки таблицы и не создает дубликаты. Пароль вводится скрыто.


In [31]:
from getpass import getpass
import psycopg

with psycopg.connect(host="localhost", port=5432, dbname="receipts_db",
                     user="postgres", password=getpass("Пароль PostgreSQL: ")) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS train_supervised_dataset (
                id BIGINT PRIMARY KEY,
                name TEXT NOT NULL CHECK (btrim(name) <> ''),
                good TEXT,
                brand TEXT
            )
        """)
        cur.execute("TRUNCATE train_supervised_dataset")
        with csv_file.open("rb") as f, cur.copy(
            "COPY train_supervised_dataset (id, name, good, brand) "
            "FROM STDIN WITH (FORMAT CSV, HEADER TRUE)"
        ) as copy:
            while chunk := f.read(1024 * 1024):
                copy.write(chunk)
        cur.execute("""
            SELECT COUNT(*) AS rows_total,
                   COUNT(DISTINCT id) AS unique_ids,
                   COUNT(*) FILTER (WHERE name IS NULL OR btrim(name) = '') AS empty_name,
                   COUNT(*) FILTER (WHERE good IS NULL OR btrim(good) = '') AS empty_good,
                   COUNT(*) FILTER (WHERE brand IS NULL OR btrim(brand) = '') AS empty_brand,
                   COUNT(*) - COUNT(DISTINCT name) AS repeated_names
            FROM train_supervised_dataset
        """)
        quality = dict(zip([c.name for c in cur.description], cur.fetchone()))
print("Таблица создана, CSV загружен.")

Таблица создана, CSV загружен.


## 3. Проверка качества данных

Проверим полноту, уникальность и повторы исходных наименований. Одинаковые `name` при разных `id` допустимы. В CSV нет даты чека, поэтому актуальность отдельных записей проверить нельзя.


In [32]:
assert len(df) == quality["rows_total"], "Часть строк CSV не загрузилась в БД"
assert quality["rows_total"] == quality["unique_ids"], "В БД есть повторяющиеся id"

print(f"Строк в CSV и БД: {len(df)} / {quality['rows_total']}")
print(f"Пустой good: {quality['empty_good']} ({quality['empty_good'] / len(df):.2%})")
print(f"Пустой brand: {quality['empty_brand']} ({quality['empty_brand'] / len(df):.2%})")
print("Без товара и бренда одновременно:", (df["good"].isna() & df["brand"].isna()).sum())


Строк в CSV и БД: 25000 / 25000
Пустой good: 819 (3.28%)
Пустой brand: 8505 (34.02%)
Без товара и бренда одновременно: 1


In [33]:
no_labels = df.loc[df[["good", "brand"]].isna().all(axis=1), ["id", "name", "good", "brand"]]
repeated = df.loc[df.duplicated("name", keep=False), ["id", "name", "good", "brand"]]
print("Строки без good и brand:\n", no_labels.to_string(index=False))
print("\nСтроки с повторяющимся name:\n", repeated.sort_values(["name", "id"]).to_string(index=False))
print("\nПолных дублей с id:", df.duplicated().sum())
print("Одинаковых name + good + brand при разных id:", df.duplicated(["name", "good", "brand"]).sum())


Строки без good и brand:
    id                       name good brand
19191 ar Club р.46-48 M цв.серый  NaN   NaN

Строки с повторяющимся name:
    id                                                                                        name       good          brand
 2475                                                              БХ Мыло хоз.Аист 72п      150г       мыло           аист
10923                                                              БХ Мыло хоз.Аист 72п      150г       мыло           аист
 5894                                         Коктейль молочный Растишка 210гр  фруктовый леденец   коктейль       растишка
13143                                         Коктейль молочный Растишка 210гр  фруктовый леденец   коктейль       растишка
 4201                                                            Корм д/кошек Friskies кролик-ово       корм       friskies
 7494                                                            Корм д/кошек Friskies кролик-ово       корм   

### Итоговый вывод

Все этапы задания реализованы в блокноте: CSV скачан с ODS.AI, таблица создана через `CREATE TABLE`, 25 000 строк загружены через `COPY`, качество проверено SQL и pandas. Число строк в БД совпадает с CSV, все `id` уникальны, пустых `name` нет.

Пустых `good` — 819 (3,28%), пустых `brand` — 8 505 (34,02%). **Одна строка (`id=19191`) не имеет ни товара, ни бренда.** Для размеченного набора это подозрительная запись, которую следует проверить отдельно; из CSV нельзя установить причину отсутствия обеих меток.

Есть **шесть пар** строк с одинаковыми `name`, `good` и `brand`, но разными `id`. Полных дублей по всем четырём полям нет. Без номера чека или даты нельзя понять, отражают ли эти пары разные покупки или повтор в наборе данных. Сами строки показаны выше.

`Last-Modified` опубликованного CSV — 1 июня 2023 года. Даты чеков в наборе нет, поэтому свежесть отдельных покупок оценить невозможно. Набор следует рассматривать как исторический срез.
